In [ ]:
# %% [markdown]
# # 🧮 Simulador ODE interactivo — Epidemia SI y Crecimiento Poblacional
# Basado en **ODEs-2.pdf (págs. 23–42)**: Problema de Valores Iniciales, modelo de
# contagio `dx/dt = r·x·(1−x)` (curva sigmoidea) y crecimiento exponencial → logístico
# `dX/dt = r·X·(1−X/K)` con capacidad de carga K.

# %%
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import odeint
from ipywidgets import interact, FloatSlider, IntSlider, Checkbox

# %% [markdown]
# ## 1) Funciones del modelo: soluciones analíticas + ODEs para integración numérica

# %%
def epidemia_x(t, x0, r):
    """Solución analítica de dx/dt = r·x·(1−x)  (PVI del resfriado común)."""
    return 1.0 / (1.0 + ((1.0 - x0) / x0) * np.exp(-r * t))

def epidemia_ode(y, t, r):
    x = y[0]
    return [r * x * (1.0 - x)]

def crecimiento_exp(t, X0, r):
    """dX/dt = r·X  →  X = X₀·e^(rt)  (curva en J)."""
    return X0 * np.exp(r * t)

def crecimiento_log(t, X0, r, K):
    """dX/dt = r·X·(1−X/K)  →  solución logística (curva en S)."""
    return K / (1.0 + (K / X0 - 1.0) * np.exp(-r * t))

def crecimiento_ode(y, t, r, K):
    X = y[0]
    return [r * X * (1.0 - X / K)]

# %% [markdown]
# ## 2) Widget: Epidemia SI (fases 1-2-3, inflexión y predicción tipo "Pregunta 1")

# %%
def plot_epidemia(x0, r, t_max, t_pred, P):
    t  = np.linspace(0, t_max, 500)
    x  = epidemia_x(t, x0, r)
    tn = np.linspace(0, t_max, 25)                       # pocos puntos: verificación numérica
    xn = odeint(epidemia_ode, [x0], tn, args=(r,))[:, 0]
    t_inf  = np.log((1 - x0) / x0) / r                   # inflexión: x = 0.5
    x_pred = epidemia_x(t_pred, x0, r)

    fig, ax = plt.subplots(figsize=(9, 5))
    ax.plot(t, x, lw=2.5, color='crimson', label='Solución analítica x(t)')
    ax.plot(tn, xn, 'o', ms=4, color='navy', alpha=.6, label='Integración numérica (odeint)')
    ax.axhline(0.5, ls=':', c='gray', alpha=.7)
    ax.axvline(t_inf, ls=':', c='gray', alpha=.7, label=f'Inflexión t≈{t_inf:.2f} (x=0.5)')
    ax.axvline(t_pred, ls='--', c='seagreen', label=f't_pred={t_pred:.0f} → x={x_pred:.6f}')
    ax.annotate('Fase 1', xy=(t_inf*0.35, 0.12), color='gray')
    ax.annotate('Fase 2', xy=(t_inf*0.95, 0.55), color='gray')
    ax.annotate('Fase 3', xy=(min(t_max*0.8, t_inf*1.8), 0.88), color='gray')
    ax.set_title('Modelo de contagio SI: dx/dt = r·x·(1−x)', fontweight='bold')
    ax.set_xlabel('Tiempo (días)'); ax.set_ylabel('Fracción infectada x')
    ax.set_ylim(0, 1.05); ax.grid(alpha=.3); ax.legend(loc='lower right')
    plt.show()
    print(f'Predicción: con x₀={x0}, r={r}, t={t_pred:.0f} → x = {x_pred:.6f} ({x_pred*100:.2f}%)')
    print(f'Casos estimados: {x_pred*P:,.0f} de {P:,.0f} personas')

interact(plot_epidemia,
         x0=FloatSlider(value=0.02, min=0.001, max=0.5, step=0.001, description='x₀:'),
         r=FloatSlider(value=0.5, min=0.05, max=1.5, step=0.05, description='r (1/día):'),
         t_max=FloatSlider(value=30, min=5, max=120, step=5, description='t máx:'),
         t_pred=FloatSlider(value=20, min=0, max=120, step=1, description='t predicción:'),
         P=IntSlider(value=1000, min=100, max=100000, step=100, description='Población:'))

# %% [markdown]
# ## 3) Widget: Crecimiento poblacional (exponencial vs logístico, capacidad K)

# %%
def plot_poblacion(X0, r, K, t_max, escala_log):
    t  = np.linspace(0, t_max, 500)
    Xe = crecimiento_exp(t, X0, r)
    Xl = crecimiento_log(t, X0, r, K)
    t_inf = np.log(K/X0 - 1)/r if K > X0 else np.nan     # inflexión: X = K/2
    t90   = np.log(9*(K/X0 - 1))/r if K > X0 else np.nan # tiempo al 90% de K

    fig, ax = plt.subplots(figsize=(9, 5))
    ax.plot(t, Xe, lw=2, color='orange', label='Exponencial X₀·e^(rt)')
    ax.plot(t, Xl, lw=2.5, color='teal', label='Logística (satura en K)')
    ax.axhline(K, ls='--', c='red', alpha=.6, label=f'K = {K:,.0f}')
    ax.axhline(K/2, ls=':', c='gray', alpha=.6, label='K/2 (inflexión)')
    if escala_log: ax.set_yscale('log')
    ax.set_title('Crecimiento poblacional: exponencial vs logístico', fontweight='bold')
    ax.set_xlabel('Tiempo t'); ax.set_ylabel('Población X')
    ax.grid(alpha=.3, which='both'); ax.legend(loc='upper left')
    plt.show()
    print(f'X exponencial en t={t_max:.0f}: {Xe[-1]:,.4e}   (paradoja del infinito)')
    print(f'X logístico   en t={t_max:.0f}: {Xl[-1]:,.2f}   (límite K={K:,.0f})')
    if np.isfinite(t_inf): print(f'Inflexión (X=K/2) en t≈{t_inf:.2f} · 90% de K en t≈{t90:.2f}')

interact(plot_poblacion,
         X0=FloatSlider(value=100, min=1, max=1000, step=1, description='X₀:'),
         r=FloatSlider(value=0.05, min=0.001, max=1.0, step=0.001, description='r:'),
         K=FloatSlider(value=10000, min=500, max=200000, step=500, description='K:'),
         t_max=FloatSlider(value=240, min=10, max=500, step=10, description='t máx:'),
         escala_log=Checkbox(value=True, description='Escala log Y'))

# %% [markdown]
# ## 4) Verificación del PVI con los ejemplos del PDF (analítico vs numérico)

# %%
t = np.linspace(0, 30, 61)
err_ep  = np.max(np.abs(epidemia_x(t, 0.02, 0.5) - odeint(epidemia_ode, [0.02], t, args=(0.5,))[:, 0]))
err_pop = np.max(np.abs(crecimiento_log(t, 100, 0.3, 5000) - odeint(crecimiento_ode, [100], t, args=(0.3, 5000))[:, 0]))
print('Error máx epidemia (analítico vs odeint):', f'{err_ep:.2e}')
print('Error máx logística  (analítico vs odeint):', f'{err_pop:.2e}')
print('Ejemplo PDF Pregunta 1 → x(20) con x₀=0.02, r=0.5 :', f'{epidemia_x(20, 0.02, 0.5):.6f}  (esperado ≈0.997717)')
print('Ejemplo PDF fermentador → X(60 min) con X₀=100, r=0.05 :', f'{crecimiento_exp(60, 100, 0.05):,.0f}  (esperado ≈2008)')